# Prompt Chaining
This notebooks covers basics of one of the multi-agentic patterns - Prompt Chaining

The main idea behind this pattern is that many agents work on one task sequentially, like as follows:

Agent 1 -> Agent 2 -> ... -> Agent n

The output of every agent serves as input for the next agent. Looks nice when the complicated task is well defined and we can decompose it into subtasks which will be resolved step by step.

The bottleneck of the pattern is that in case for some reasons any of agents generates bad response, hallucinates or breaks then the whole chain breaks.

In [ ]:
from langchain_groq.chat_models import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field, SecretStr

In [2]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    GROQ_API_KEY = SecretStr(api_key)

In [3]:
base_llm = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

**Business take**:
We are building a system that will take unstructured device description and convert into a structured json object with specifications and their values. Additionally the system has to rate the device, provide cons and prons. 

The problem is well defined and can be decomposed into pretty deterministic steps:
1. Extract specifications
2. Rate

So here we can easily implement prompt chaining pattern with 2 agents: `Extractor` and `Reviewer`

In [ ]:
extractor_prompt = ChatPromptTemplate.from_template("You are an intelligent AI assistant aimed at information extraction. " \
"Given a device description, extract as many technical specifications as possible" \
"device description: {description}")

In [ ]:
reviewer_prompt = ChatPromptTemplate.from_template("You are an intelligent AI assistant designed to rate device by it's specs" \
"Given technical specifications of a device you should give a user your own opinion on whether the divice is good or not" \
"specs:{specs}")

Additionally as one of our steps involves structured output, we define neccessary schemas.

In [ ]:
class Specification(BaseModel):
    spec: dict[str, str] = Field(default={}, description="A dict-like single specification where key is a name of specification(e.g. Refresh Rate, Voltage) and value is its exact numeric value(e.g. 144Hz, 20V) if provided")

In [7]:
class ExtractorResponse(BaseModel):
    specs: list[Specification] = Field(default=[], description="A list of Specifications exctracted form a provided device description")

In [ ]:
class ReviewerResponse(BaseModel):
    advantages: list[str] = Field(default=[], description="A list of device advantages based on its specifications")
    disadvantages: list[str] = Field(default=[], description="A list of device disadvantages based on its specifications")
    review: str = Field(default="", description="A final review of a device based on its advantages and disadvantages")

In [9]:
extractor_chain = extractor_prompt | base_llm.with_structured_output(ExtractorResponse)

In [10]:
reviewer_chain = reviewer_prompt | base_llm.with_structured_output(ReviewerResponse)

In [11]:
input_query = "LG Monitor 144Hz Refresh Rate 24.1 inch diagonal OLED"

In [12]:
extractor_response = extractor_chain.invoke({
    "description": input_query
})
extractor_response

ExtractorResponse(specs=[Specification(spec={'Refresh Rate': '144Hz'}), Specification(spec={'Diagonal': '24.1 inch'}), Specification(spec={'Panel Type': 'OLED'})])

In [13]:
reviewer_response = reviewer_chain.invoke({
    "specs": extractor_response
})
reviewer_response

ReviewerResponse(advantages=['High refresh rate for smooth visuals', '24.1 inch display size is suitable for most users', 'OLED panel for good color accuracy'], disadvantages=['Refresh rate might be lower for users requiring 240Hz or higher'], review='This device has a good balance of features, but could be improved with a higher refresh rate.')

In [14]:
reviewer_response.model_dump()

{'advantages': ['High refresh rate for smooth visuals',
  '24.1 inch display size is suitable for most users',
  'OLED panel for good color accuracy'],
 'disadvantages': ['Refresh rate might be lower for users requiring 240Hz or higher'],
 'review': 'This device has a good balance of features, but could be improved with a higher refresh rate.'}